In [1]:
import boto3
import pandas as pandas
from sqlalchemy import create_engine
from io import StringIO
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd

## Let's define our globals
# load .env variables
load_dotenv()
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
REGION = 'us-east-2a'

# track our RDS DB details
DB_HOST ='olist-db.ctkoee6m63v2.us-east-2.rds.amazonaws.com'
DB_PORT = '5432'
DB_NAME = 'postgres'
DB_USER = 'postgres'
DB_PASSWORD = os.getenv('DB_PASSWORD')

# s3 details
BUCKET_NAME = 'olist-data-3482-3050'

FILES_TO_LOAD = {
    'olist_orders_dataset.csv': 'staging_orders',
    'olist_customers_dataset.csv': 'staging_customers',
    'olist_order_items_dataset.csv': 'staging_items',
    'olist_products_dataset.csv': 'staging_products',
    'olist_sellers_dataset.csv': 'staging_sellers',
    'olist_geolocation_dataset.csv': 'staging_geolocation',
    'olist_order_payments_dataset.csv': 'staging_payments',
    'olist_order_reviews_dataset.csv': 'staging_reviews',
    'product_category_name_translation.csv': 'staging_category_translation'
}

## Now we need to connect to our two AWS interfaces

In [ ]:
# connect to our S3 buckets
s3 = boto3.client('s3',
    aws_access_key_id = AWS_ACCESS_KEY_ID,
    aws_secret_access_key = AWS_SECRET_ACCESS_KEY,
)

response = s3.list_buckets()

print('Buckets:')
for bucket in response['Buckets']:
    print(f"* {bucket['Name']}")

#pull csvs from s3
olist_bucket = s3.list_objects(Bucket=BUCKET_NAME)['Contents']
print(f'\nItems in {BUCKET_NAME}:')
for item in olist_bucket:
    print(item['Key'])


# connect to rds
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def load_data(filename, tablename) -> None:
    """
    load_data: takes in a filename and its respective tablename, downloads that csv from an S3 bucket,
    then converts it to a pd.DataFrame and uploads it to an RDS database with the table name tablename
    """

    print(f'\nDownloading {filename} from S3...')
    response = s3.get_object(Bucket=BUCKET_NAME, Key="olist-data/" + filename)
    
    df = pd.read_csv(response['Body'])

    print(f'Found rows: {len(df)}')
    print(f'Attaching to table: {tablename}')

    # write to RDS
    try:
        print(f'Attempting to push {tablename} to RDS database...')
        df.to_sql(tablename, con=engine, if_exists='replace', index=False)
    except Exception as e:
        print('Unable to push CSV to RDS database, exceptions: {e}')

    print(f'Succesfully pushed: {tablename}')


Buckets:
* olist-data-3482-3050

Items in olist-data-3482-3050:
olist-data/
olist-data/olist_customers_dataset.csv
olist-data/olist_geolocation_dataset.csv
olist-data/olist_order_items_dataset.csv
olist-data/olist_order_payments_dataset.csv
olist-data/olist_order_reviews_dataset.csv
olist-data/olist_orders_dataset.csv
olist-data/olist_products_dataset.csv
olist-data/olist_sellers_dataset.csv
olist-data/product_category_name_translation.csv


In [3]:
# if name == main:
print('Beginning Ingestion Pipeline...')
df_list = [] 

for filename, tablename in FILES_TO_LOAD.items():
    try:
        df_list.append(load_data(filename, tablename))
    except Exception as e:
        print(f"Couldn't download {filename} and upload it to the RDS DB.")

print(f'Number of tables pushed: {len(df_list)}') # should be 9
print('--- Completed Ingestion Pipeline! ---')

Beginning Ingestion Pipeline...

Found rows: 99441
Attaching to table: staging_orders
Attempting to push staging_orders to RDS database...
Succesfully pushed: staging_orders

Found rows: 99441
Attaching to table: staging_customers
Attempting to push staging_customers to RDS database...
Succesfully pushed: staging_customers

Found rows: 112650
Attaching to table: staging_items
Attempting to push staging_items to RDS database...
Succesfully pushed: staging_items

Found rows: 32951
Attaching to table: staging_products
Attempting to push staging_products to RDS database...
Succesfully pushed: staging_products

Found rows: 3095
Attaching to table: staging_sellers
Attempting to push staging_sellers to RDS database...
Succesfully pushed: staging_sellers

Found rows: 1000163
Attaching to table: staging_geolocation
Attempting to push staging_geolocation to RDS database...
Succesfully pushed: staging_geolocation

Found rows: 103886
Attaching to table: staging_payments
Attempting to push staging_